In [1]:
# ── Cell 0: Environment setup ─────────────────────────────────────────────────
import os, torch, shutil
from pathlib import Path

# download_data.py places EK100_MIR/ next to src/ by default.
# Override with: export EK100_MIR_ROOT=/your/path
_here = Path(os.getcwd())
EK100_MIR_ROOT = Path(os.environ.get("EK100_MIR_ROOT", _here.parent / "EK100_MIR"))

assert EK100_MIR_ROOT.exists(), (
    f"EK100_MIR not found at {EK100_MIR_ROOT}\n"
    "Run  python src/download_data.py  first, or set EK100_MIR_ROOT env var."
)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
    print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
else:
    print("No GPU — inference will run on CPU (may be slow)")

_, _, free = shutil.disk_usage(EK100_MIR_ROOT)
print(f"Disk free : {free/1e9:.0f} GB")
print(f"EK100_MIR : {EK100_MIR_ROOT.resolve()}")

GPU : NVIDIA GeForce RTX 5070  (12.3 GB VRAM)
CUDA: 13.0 | PyTorch: 2.11.0+cu130
Disk free : 1409 GB
EK100_MIR : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR


In [2]:
# ── Cell 2: Clone JPoSE repo ──────────────────────────────────────────────────
import subprocess

REPOS_DIR = EK100_MIR_ROOT / "repos"
REPOS_DIR.mkdir(exist_ok=True)
JPOSE_DIR = REPOS_DIR / "Joint-Part-of-Speech-Embeddings"

if not JPOSE_DIR.exists():
    r = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/mwray/Joint-Part-of-Speech-Embeddings.git",
         str(JPOSE_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{r.stderr}")
    print(f"Cloned : {JPOSE_DIR}")
else:
    print(f"Already exists: {JPOSE_DIR}")

Cloned : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings


In [3]:
# ── Cell 3: Configure paths ───────────────────────────────────────────────────
import re, pickle, shutil
import numpy as np
from pathlib import Path

# Repo (source code)
JPOSE_DIR = EK100_MIR_ROOT / "repos" / "Joint-Part-of-Speech-Embeddings"

# Data extracted by download_data.py to EK100_MIR/data/JPoSE/
# The zip has a data/ prefix, so files land at EK100_MIR/data/JPoSE/data/
JPOSE_DATA_ROOT = EK100_MIR_ROOT / "data" / "JPoSE" / "data"

# The repo ships an empty data/ dir with .gitkeep placeholders.
# Remove it and replace with a symlink to the downloaded data.
data_link = JPOSE_DIR / "data"
if data_link.is_symlink():
    print(f"Symlink exists: {data_link}")
elif data_link.is_dir():
    shutil.rmtree(data_link)
    data_link.symlink_to(JPOSE_DATA_ROOT)
    print(f"Replaced git placeholder with symlink: {data_link} -> {JPOSE_DATA_ROOT}")
else:
    data_link.symlink_to(JPOSE_DATA_ROOT)
    print(f"Symlinked: {data_link} -> {JPOSE_DATA_ROOT}")

MODELS_DIR     = JPOSE_DIR / "data" / "models"
VID_FEAT_DIR   = JPOSE_DIR / "data" / "video_features"
TXT_FEAT_DIR   = JPOSE_DIR / "data" / "text_features"
DATAFRAMES_DIR = JPOSE_DIR / "data" / "dataframes"
RELATIONAL_DIR = JPOSE_DIR / "data" / "relational"
RELEVANCY_DIR  = JPOSE_DIR / "data" / "relevancy"

SUBMISSIONS_DIR = EK100_MIR_ROOT / "submissions"
ZIPS_DIR        = EK100_MIR_ROOT / "submission_zips"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths OK")
print(f"  JPoSE repo  : {JPOSE_DIR}")
print(f"  Data root   : {JPOSE_DATA_ROOT}")
print(f"  Submissions : {SUBMISSIONS_DIR}")
print(f"  ZIPs        : {ZIPS_DIR}")

Replaced git placeholder with symlink: /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings/data -> /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/data/JPoSE/data
Paths OK
  JPoSE repo  : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings
  Data root   : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/data/JPoSE/data
  Submissions : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submissions
  ZIPs        : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submission_zips


In [4]:
# ── Cell 4: Verify data ───────────────────────────────────────────────────────
def check_path(path, label, required=True):
    p = Path(path)
    if p.is_dir():
        files  = [f for f in p.rglob("*") if f.is_file()]
        ok     = bool(files)
        detail = f"{len(files)} file(s) ({sum(f.stat().st_size for f in files)/1e6:.0f} MB)"
    else:
        ok     = p.exists()
        detail = f"{p.stat().st_size/1e9:.2f} GB" if ok else "NOT FOUND"
    icon = "OK" if ok else ("ERR" if required else "WARN")
    print(f"  [{icon}]  {label}: {detail}")
    return ok

print("── JPoSE / MLP data ────────────────────────────────────")
all_ok = all([
    check_path(MODELS_DIR,     "models"),
    check_path(VID_FEAT_DIR,   "video_features"),
    check_path(TXT_FEAT_DIR,   "text_features"),
    check_path(DATAFRAMES_DIR, "dataframes"),
    check_path(RELATIONAL_DIR, "relational"),
    check_path(RELEVANCY_DIR,  "relevancy"),
])

checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
check_path(checkpoint, "JPoSE_BEST checkpoint")

if not all_ok:
    raise RuntimeError("Missing data — run:  python src/download_data.py")
print("\nAll data OK")

── JPoSE / MLP data ────────────────────────────────────
  [OK]  models: 5 file(s) (12 MB)
  [OK]  video_features: 6 file(s) (1955 MB)
  [OK]  text_features: 10 file(s) (107 MB)
  [OK]  dataframes: 7 file(s) (20 MB)
  [OK]  relational: 4 file(s) (3 MB)
  [OK]  relevancy: 2 file(s) (93 MB)
  [OK]  JPoSE_BEST checkpoint: 0.01 GB

All data OK


In [5]:
# ── Cell 5: Apply patches (torch.load weights_only) ───────────────────────────
for fpath in JPOSE_DIR.joinpath("src").rglob("*.py"):
    txt = fpath.read_text()
    new = re.sub(
        r"torch\.load\(([^,)]+)\)",
        r"torch.load(\1, weights_only=False)",
        txt,
    )
    if new != txt:
        fpath.write_text(new)
        print(f"  Patched: {fpath.name}")

print("Patches OK")

Patches OK


In [6]:
# ── Cell 6: Inspect checkpoint ────────────────────────────────────────────────
import torch

MODEL_NAME = "JPoSE_BEST"
COMB_FUNC  = "cat"

CHECKPOINT = MODELS_DIR / MODEL_NAME / "model" / f"EPIC_100_retrieval_{MODEL_NAME}.pth"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

state = torch.load(str(CHECKPOINT), weights_only=False, map_location="cpu")
print(f"Checkpoint: {CHECKPOINT.name}")
print(f"Layers ({len(state)} tensors):")
for name, tensor in state.items():
    print(f"  {name:50s}  {str(tuple(tensor.shape)):20s}  {tensor.dtype}")

Checkpoint: EPIC_100_retrieval_JPoSE_BEST.pth
Layers (36 tensors):
  MMENs.verb.SMEs.t.fc.weight                         (256, 200)            torch.float32
  MMENs.verb.SMEs.t.fc.bias                           (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.fc.weight                      (256, 256)            torch.float32
  MMENs.verb.SMEs.t.cg.fc.bias                        (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.batch_norm.weight              (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.batch_norm.bias                (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.batch_norm.running_mean        (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.batch_norm.running_var         (256,)                torch.float32
  MMENs.verb.SMEs.t.cg.batch_norm.num_batches_tracked  ()                    torch.int64
  MMENs.verb.SMEs.v.fc.weight                         (256, 3072)           torch.float32
  MMENs.verb.SMEs.v.fc.bias       

In [7]:
# ── Cell 7: JPoSE inference ───────────────────────────────────────────────────
import subprocess

jpose_out = SUBMISSIONS_DIR / f"{MODEL_NAME}_test_latest.pkl"

print(f"Running JPoSE inference ({MODEL_NAME}, comb-func={COMB_FUNC})...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python src/train/test_jpose_triplet.py "{CHECKPOINT}" '
    f'--comb-func {COMB_FUNC} --challenge-submission "{jpose_out}" 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-600:])
    raise RuntimeError("JPoSE inference failed")

assert jpose_out.exists(), f"Output not generated: {jpose_out}"
with open(jpose_out, "rb") as f:
    sub = pickle.load(f)

sim_mat = np.array(sub["sim_mat"], dtype=np.float32)
vis_ids = list(sub["vis_ids"])
txt_ids = list(sub["txt_ids"])

assert sim_mat.shape == (9668, 3842), f"Unexpected shape: {sim_mat.shape}"
print(f"\nsim_mat : {sim_mat.shape}  dtype={sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids: {len(txt_ids)}")
print(f"Saved   : {jpose_out.name}  ({jpose_out.stat().st_size/1e6:.1f} MB)")

Running JPoSE inference (JPoSE_BEST, comb-func=cat)...
/home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings/src/models/mmen.py:12: SyntaxWarning: invalid escape sequence '\#'
  modality_dict: {modality: {'layer_sizes': [], 'num_layers': \#}}
/home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings/src/evaluation/nDCG.py:7: SyntaxWarning: invalid escape sequence '\s'
  DCG = \sum_{i=1}^k \frac{rel_i}{log_2(i + 1)}
/home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/repos/Joint-Part-of-Speech-Embeddings/src/evaluation/mAP.py:8: SyntaxWarning: invalid escape sequence '\s'
  \frac{\sum_{k=1}^n p(k) x rel(k)}{num_rel_docs}
Namespace(batch_size=64, checkpoint_rate=10, embedding_size=256, gpu=False, learning_rate=0.01, margin=1.0, momentum=0.9, num_epochs=100, num_layers=2, optimiser='SGD', out_dir='./logs/runs', tt_weight=1.0, tv_weight=2.0, vt_weight=1.0, vv_weight=1.0, action_weight=1.0, comb_f

In [8]:
# ── Cell 8: Create submission ZIP ─────────────────────────────────────────────
SLS_PT = 2
SLS_TL = 3
SLS_TD = 3
SUBMISSION_NAME = f"{MODEL_NAME}_submission"

def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):
    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   np.array(sim_mat, dtype=np.float32),
        "vis_ids":   [str(v) for v in vis_ids],
        "txt_ids":   [str(t) for t in txt_ids],
    }
    raw = pickle.dumps(payload, protocol=2)
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

tmp_pkl = Path("/tmp/test.pkl")
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
tmp_pkl.write_bytes(pkl_bytes)

check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == (9668, 3842)
print(f"test.pkl OK: {len(pkl_bytes)/1e6:.1f} MB  |  sim_mat={np.array(check['sim_mat']).shape}")

zip_path = ZIPS_DIR / f"{SUBMISSION_NAME}.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell=True, check=True, capture_output=True,
)

print(f"ZIP created : {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print(f"Location    : {zip_path}")
print(f"\nReady to upload to: https://www.codabench.org/competitions/12008")

test.pkl OK: 218.1 MB  |  sim_mat=(9668, 3842)
ZIP created : JPoSE_BEST_submission.zip  (155.6 MB)
Location    : /home/nico/Desktop/Multi-Instance-Retrieval-EK-100/EK100_MIR/submission_zips/JPoSE_BEST_submission.zip

Ready to upload to: https://www.codabench.org/competitions/12008
